# Semana 9: Procesamiento con Spark y EDA

## Impacto ambiental de la energia

Este notebook aplica la pauta de Semana 9 a los registros consolidados en MongoDB por `main.py`. Para evitar comparaciones invalidas entre unidades, el EDA relacional usa exclusivamente los datasets EIA que comparten region y periodo:

- `annual_generation_state`: generacion electrica en MWh.
- `emission_annual`: emisiones en toneladas metricas de CO2.

La caracteristica principal creada es `intensidad_co2 = emisiones_co2_mt / generacion_mwh`, medida en toneladas CO2/MWh.

## Objetivos de la actividad

1. Leer los datos desde MongoDB con Spark, usando el CSV de Semana 7 como respaldo reproducible.
2. Limpiar duplicados, nulos y valores no validos.
3. Crear variables: intensidad de CO2, participacion en generacion, transformaciones logaritmicas y puntaje z.
4. Examinar estadisticas descriptivas, valores faltantes, sesgo y outliers por rango intercuartil.
5. Calcular matrices Pearson y Spearman y producir visualizaciones.

In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display, Image

# Para forzar la fuente CSV local durante pruebas, descomenta la siguiente linea:
# os.environ['FUENTE_DATOS'] = 'csv'

OUTPUT_DIR = Path('salidas')
print('Fuente preferida:', os.getenv('FUENTE_DATOS', 'mongo'))

## Ejecucion del EDA con PySpark

El script asociado contiene la conexion a MongoDB, limpieza, ingenieria de caracteristicas, correlaciones y exportacion de resultados. Los mensajes impresos muestran conteos, resumen estadistico, cuartiles y asimetria.

In [ ]:
%run ./eda_semana9.py

## Tabla analitica generada

Cada fila representa una combinacion region-categoria energetica del periodo comun. Esta tabla es la entrada controlada para el aprendizaje no supervisado de Semana 10.

In [ ]:
features = pd.read_csv(OUTPUT_DIR / 'features_eda_semana9.csv')
display(features.head(10))
display(features[['generacion_mwh', 'emisiones_co2_mt', 'intensidad_co2', 'participacion_generacion']].describe())

## Correlaciones

- Pearson mide asociacion lineal entre las magnitudes.
- Spearman se calcula sobre rangos y permite contrastar relaciones monotonicas ante asimetria y valores extremos.

In [ ]:
pearson = pd.read_csv(OUTPUT_DIR / 'correlacion_pearson.csv', index_col=0)
spearman = pd.read_csv(OUTPUT_DIR / 'correlacion_spearman.csv', index_col=0)
print('Pearson')
display(pearson.round(4))
print('Spearman')
display(spearman.round(4))

## Visualizaciones producidas

In [ ]:
for nombre in ['boxplot_intensidad_categoria.png', 'dispersion_generacion_emisiones.png', 'correlacion_pearson.png']:
    display(Image(filename=str(OUTPUT_DIR / 'figuras' / nombre)))

## Interpretacion sugerida

- Una intensidad CO2 elevada identifica categorias/regiones con mayor impacto por unidad de electricidad generada.
- La dispersion logaritmica evita que las regiones de mayor volumen oculten los patrones del resto.
- Diferencias importantes entre Pearson y Spearman indican que la relacion esta influida por distribuciones sesgadas u outliers.
- Los registros marcados como outlier no se eliminan automaticamente: pueden representar casos energeticos relevantes para investigar.